# Scala + Apache Spark: Topic-wise Notebook (runs on Google Colab)

Colab has no built-in Scala kernel, so this notebook uses a Python kernel and runs each Scala program with **scala-cli** (installed in step 0).
Each topic = one cell that writes a `.scala` file (`%%writefile`) + one cell that runs it.

**How to use:** Runtime -> Run all. Run the setup cell first.

## 0. Setup (Java 11 + scala-cli)

In [ ]:
%%bash
apt-get -qq install -y openjdk-11-jdk-headless > /dev/null
update-alternatives --set java /usr/lib/jvm/java-11-openjdk-amd64/bin/java
curl -fL https://github.com/VirtusLab/scala-cli/releases/latest/download/scala-cli-x86_64-pc-linux.gz | gzip -d > /usr/local/bin/scala-cli
chmod +x /usr/local/bin/scala-cli
java -version
scala-cli version

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
# Quick test
!scala-cli run --server=false -S 2.13.12 -e 'println("Scala is working!")'

## 1. Functions, Variables, Strings, User Input, Numbers, Types, Operators, Booleans

In [ ]:
%%writefile basics.scala
//> using scala 2.13.12

object Basics {

  // ---------- FUNCTIONS (expanded format) ----------
  // def name(param: Type, ...): ReturnType = { body }   <- last expression is the return value
  def add(a: Int, b: Int): Int = {
    val result = a + b
    result                       // no 'return' keyword needed
  }

  def greet(name: String = "Colab"): String = s"Hello, $name!"   // default parameter
  def sumAll(nums: Int*): Int = nums.sum                         // varargs
  def multiply(a: Int)(b: Int): Int = a * b                      // multiple parameter lists (currying)
  def applyTwice(f: Int => Int, x: Int): Int = f(f(x))           // higher-order function

  def main(args: Array[String]): Unit = {

    println("---- Functions ----")
    println(add(2, 3))
    println(greet())                       // uses default
    println(greet(name = "Rui"))           // named argument
    println(sumAll(1, 2, 3, 4))
    println(multiply(3)(4))
    val square = (x: Int) => x * x         // anonymous function (lambda)
    println(applyTwice(square, 3))         // (3^2)^2 = 81

    // ---------- VARIABLES ----------
    println("---- Variables ----")
    val pi = 3.14159       // val = immutable (cannot be reassigned)
    var counter = 0        // var = mutable
    counter += 1
    println(s"pi=$pi, counter=$counter")
    val inferred = 42      // type inferred as Int
    val explicit: Double = 42  // explicit type

    // ---------- STRINGS ----------
    println("---- Strings ----")
    val s = "Hello Scala"
    println(s.length)                       // 11
    println(s.toUpperCase)
    println(s.substring(0, 5))              // Hello
    println(s.replace("Scala", "Spark"))
    println(s.split(" ").toList)            // List(Hello, Scala)
    println(s"Interpolation: 2 + 3 = ${2 + 3}")   // s-interpolator
    println(f"Formatted: $pi%.2f")                // f-interpolator
    println("Multi" + "-" + "line")

    // ---------- GETTING USER INPUT ----------
    // In Colab '!' commands cannot read the keyboard, so we SIMULATE input.
    // On a real terminal just call: val name = scala.io.StdIn.readLine()
    println("---- User Input ----")
    val fake = new java.io.StringReader("Rui\n25\n")
    Console.withIn(fake) {
      val name = scala.io.StdIn.readLine()   // reads a line
      val age  = scala.io.StdIn.readInt()    // reads an Int
      println(s"Name=$name, Age=$age")
    }

    // ---------- NUMBERS ----------
    println("---- Numbers ----")
    val i: Int = 10                 // 32-bit
    val l: Long = 10000000000L      // 64-bit
    val d: Double = 3.5             // 64-bit decimal
    val f: Float = 2.5f             // 32-bit decimal
    val big = BigInt("123456789012345678901234567890")
    println(big * 2)
    println(10 / 3)                 // 3   (integer division)
    println(10.0 / 3)               // 3.3333...
    println(10 % 3)                 // 1   (remainder)
    println(math.sqrt(16))          // 4.0
    println(math.pow(2, 10))        // 1024.0
    println(math.max(3, 7))

    // ---------- VARIABLE TYPES ----------
    println("---- Types ----")
    val b: Byte = 127
    val sh: Short = 32000
    val c: Char = 'A'
    val str: String = "text"
    val bool: Boolean = true
    val unit: Unit = ()             // Unit ~ void
    val any: Any = 5                // Any = parent of everything
    println(any.isInstanceOf[Int])  // true
    println(d.toInt)                // 3  (type conversion)
    println("123".toInt + 1)        // 124
    println(c.toInt)                // 65
    val opt: Option[Int] = None     // Option = safe "maybe a value"
    println(opt.getOrElse(0))

    // ---------- OPERATORS ----------
    println("---- Operators ----")
    val x = 7; val y = 3
    println(s"Arithmetic: ${x + y} ${x - y} ${x * y} ${x / y} ${x % y}")
    println(s"Relational: ${x > y} ${x < y} ${x == y} ${x != y} ${x >= y}")
    println(s"Bitwise: ${x & y} ${x | y} ${x ^ y} ${x << 1} ${x >> 1}")
    var z = 5; z += 2; z *= 3          // assignment operators
    println(z)
    println(x.+(y))                    // operators are really METHODS: x + y == x.+(y)

    // ---------- BOOLEANS ----------
    println("---- Booleans ----")
    val t = true; val fl = false
    println(t && fl)    // AND -> false
    println(t || fl)    // OR  -> true
    println(!t)         // NOT -> false
    val age = 20
    if (age >= 18 && age < 60) println("Adult") else println("Other")
    println(if (age > 18) "yes" else "no")   // if is an expression
  }
}

In [ ]:
!scala-cli run --server=false basics.scala

## 2. Data Structures: Array, List, Tuple, Set, HashSet, Map + Functional Combinators

In [ ]:
%%writefile collections.scala
//> using scala 2.13.12

import scala.collection.mutable

object Collections {
  def main(args: Array[String]): Unit = {

    // ---------- ARRAY ---------- (fixed size, mutable elements, index from 0)
    println("---- Array ----")
    val arr = Array(10, 20, 30, 40)
    arr(0) = 99                          // elements can be changed
    println(arr.mkString(", "))
    println(arr.length)
    val arr2 = new Array[Int](3)         // [0,0,0]
    val range = (1 to 5).toArray         // 1,2,3,4,5

    // ---------- LIST ---------- (immutable linked list)
    println("---- List ----")
    val list = List(1, 2, 3, 4, 5)
    println(list.head)                   // 1
    println(list.tail)                   // List(2,3,4,5)
    val list2 = 0 :: list                // prepend -> new list
    val joined = list ++ List(6, 7)      // concatenate
    println(list2); println(joined)
    println(list.reverse)

    // ---------- TUPLE ---------- (fixed size, mixed types)
    println("---- Tuple ----")
    val person = ("Rui", 25, true)
    println(person._1)                   // Rui
    println(person._2)                   // 25
    val (name, age, flag) = person       // destructuring
    println(s"$name $age $flag")

    // ---------- SET ---------- (unique elements, no order guarantee)
    println("---- Set ----")
    val set = Set(1, 2, 2, 3, 3)         // duplicates removed
    println(set)
    println(set.contains(2))
    println(set + 10)                    // new set with 10 added
    println(Set(1, 2, 3) & Set(2, 3, 4)) // intersection
    println(Set(1, 2) | Set(2, 3))       // union

    // ---------- HASH SET ---------- (mutable, hash based, fast lookup)
    println("---- HashSet ----")
    val hs = mutable.HashSet[String]()
    hs += "apple"; hs += "banana"; hs += "apple"   // adds in place
    println(hs)                          // apple only once
    hs -= "banana"
    println(hs.size)

    // ---------- MAP ---------- (key -> value)
    println("---- Map ----")
    val m = Map("a" -> 1, "b" -> 2, "c" -> 3)        // immutable
    println(m("a"))                                   // 1
    println(m.get("z"))                               // None (safe access)
    println(m.getOrElse("z", 0))
    val m2 = m + ("d" -> 4)                           // new map
    println(m2.keys); println(m2.values)
    val mm = mutable.Map("x" -> 1)                    // mutable map
    mm("y") = 2; mm += ("z" -> 3)
    println(mm)
    for ((k, v) <- m) println(s"$k => $v")

    // ---------- FUNCTIONAL COMBINATORS ----------
    println("---- Functional Combinators ----")
    val nums = List(1, 2, 3, 4, 5, 6)

    // MAP: apply a function to every element -> same size
    println(nums.map(x => x * 2))                     // List(2,4,6,8,10,12)
    println(nums.map(_ + 1))                          // '_' shorthand

    // FILTER: keep elements satisfying a predicate
    println(nums.filter(_ % 2 == 0))                  // List(2,4,6)

    // FLATMAP: map then flatten
    println(List("hi there", "scala fun").flatMap(_.split(" ")))

    // FOLD / REDUCE: combine all elements into one value
    println(nums.reduce(_ + _))                       // 21
    println(nums.foldLeft(100)(_ + _))                // 121 (starts from 100)

    // FOREACH: side effects (returns Unit)
    nums.take(3).foreach(println)

    // Others
    println(nums.zip(List("a", "b", "c")))            // pairs up
    println(nums.partition(_ > 3))                    // (List(4,5,6), List(1,2,3))
    println(nums.groupBy(_ % 2 == 0))                 // Map(false->..., true->...)
    println(nums.exists(_ > 5))                       // true
    println(nums.forall(_ > 0))                       // true
    println(nums.sortWith(_ > _))                     // descending
    println(nums.sum + " " + nums.max + " " + nums.min)

    // MAP combinators on Map
    println(m.map { case (k, v) => (k, v * 10) })
    println(m.filter { case (_, v) => v > 1 })
    println(m.mapValues(_ + 1).toMap)

    // Word count - classic combinator pipeline
    val words = List("spark", "scala", "spark", "big", "data", "spark")
    println(words.groupBy(identity).map { case (w, ws) => (w, ws.size) })
  }
}

In [ ]:
!scala-cli run --server=false collections.scala

## 3. Classes & Objects: Class, Singleton, Anonymous object, Companion, Case Class, Constructors, Overloading, `this`

In [ ]:
%%writefile oop1.scala
//> using scala 2.13.12

// ---------- CLASS + PRIMARY CONSTRUCTOR ----------
// Parameters after the class name = primary constructor. 'val' makes them public fields.
class Person(val name: String, var age: Int) {

  println(s"Constructor called for $name")      // class body runs on object creation

  // ---------- AUXILIARY CONSTRUCTOR ---------- (must call another constructor first)
  def this(name: String) = {
    this(name, 0)                                // 'this(...)' calls the primary constructor
  }

  // ---------- 'this' KEYWORD ---------- refers to the current object
  def setAge(age: Int): this.type = {            // param 'age' shadows field, so use this.age
    this.age = age
    this                                         // returning 'this' allows chaining
  }

  // ---------- METHOD OVERLOADING ---------- same name, different parameters
  def greet(): String = s"Hi, I am $name"
  def greet(other: String): String = s"Hi $other, I am $name"
  def greet(other: String, times: Int): String = (greet(other) + " ") * times

  override def toString: String = s"Person($name, $age)"
}

// ---------- COMPANION OBJECT ----------
// Same name + same file as the class. Holds "static"-like members and factory methods.
class Account private (val id: Int, private var balance: Double) {   // private constructor
  def deposit(x: Double): Unit = balance += x
  def getBalance: Double = balance
}
object Account {
  private var counter = 0
  def apply(initial: Double): Account = {        // apply lets you write Account(100.0) without 'new'
    counter += 1
    new Account(counter, initial)                // companion can call the private constructor
  }
  def totalCreated: Int = counter
}

// ---------- SINGLETON OBJECT ---------- only ONE instance, no 'new'
object Config {
  val appName = "ScalaDemo"
  def hello(): Unit = println(s"Welcome to $appName")
}

// ---------- CASE CLASS ----------
// Auto-generates: apply, toString, equals, hashCode, copy, pattern matching, getters.
case class Point(x: Int, y: Int)

object OOP1 {
  def main(args: Array[String]): Unit = {

    println("---- Class / Constructors / this ----")
    val p1 = new Person("Alice", 30)             // primary
    val p2 = new Person("Bob")                   // auxiliary
    println(p1); println(p2)
    p2.setAge(20).setAge(21)                     // chaining thanks to 'this'
    println(p2)

    println("---- Method Overloading ----")
    println(p1.greet())
    println(p1.greet("Carol"))
    println(p1.greet("Dave", 2))

    println("---- Singleton Object ----")
    Config.hello()
    println(Config.appName)

    println("---- Anonymous object ----")
    // Object created without naming a class
    val anon = new { val id = 1; def say() = println("I am anonymous") }
    import scala.language.reflectiveCalls
    anon.say()
    // Anonymous subclass of a class/trait
    val anonPerson = new Person("Anon", 1) { override def toString = "Anonymous subclass!" }
    println(anonPerson)

    println("---- Companion Object ----")
    val a1 = Account(100.0)                      // calls Account.apply
    val a2 = Account(500.0)
    a1.deposit(50)
    println(a1.getBalance)                       // 150.0
    println(Account.totalCreated)                // 2

    println("---- Case Class ----")
    val pt = Point(1, 2)                         // no 'new' needed
    println(pt)                                  // Point(1,2)
    println(pt == Point(1, 2))                   // true (structural equality)
    val pt2 = pt.copy(y = 10)                    // copy with a changed field
    println(pt2)
    pt2 match {                                  // pattern matching
      case Point(0, _) => println("on y-axis")
      case Point(x, y) => println(s"x=$x y=$y")
    }
  }
}

In [ ]:
!scala-cli run --server=false oop1.scala

## 4. Inheritance, Field Overriding, Final, Abstract Class, Trait, Mixins, Access Modifiers

In [ ]:
%%writefile oop2.scala
//> using scala 2.13.12

// ---------- ABSTRACT CLASS ---------- cannot be instantiated; may have abstract + concrete members
abstract class Animal(val name: String) {
  val sound: String                       // abstract field (no value)
  def speak(): Unit                       // abstract method (no body)
  def eat(): Unit = println(s"$name is eating")   // concrete method
}

// ---------- INHERITANCE ---------- 'extends'
class Dog(name: String) extends Animal(name) {
  // ---------- FIELD OVERRIDING ---------- 'override val' replaces a parent's field
  override val sound: String = "Woof"
  def speak(): Unit = println(s"$name says $sound")
  override def eat(): Unit = {                    // method overriding
    super.eat()                                   // call the parent's version
    println(s"$name loves bones")
  }
}

class Parent {
  val kind: String = "parent"
  def info(): String = s"I am a $kind"
}
class Child extends Parent {
  override val kind: String = "child"             // overridden field
}

// ---------- FINAL ----------
class Base {
  final val MAX = 100                             // final val: cannot be overridden
  final def locked(): String = "cannot override me"
  def open(): String = "can override me"
}
final class Sealed                                 // final class: cannot be extended
// class X extends Sealed   // <- would NOT compile

// ---------- TRAIT ---------- like interface + can hold code; a class can mix in many
trait Logger {
  def log(msg: String): Unit = println(s"[LOG] $msg")
}
trait Timestamp {
  def now(): String = "2026-01-01 10:00"
}
trait Shape {
  def area(): Double                              // abstract method in a trait
  def describe(): String = s"Area = ${area()}"
}

// ---------- TRAIT MIXINS ---------- 'extends' first trait/class, then 'with' for more
class Circle(r: Double) extends Shape with Logger with Timestamp {
  def area(): Double = math.Pi * r * r
  def run(): Unit = log(s"${now()} -> ${describe()}")
}

// ---------- ACCESS MODIFIERS ----------
class Secrets {
  val publicField = "everyone can see"            // default = public
  protected val protectedField = "me + subclasses"
  private val privateField = "only this class"
  private[this] val objectPrivate = "only this very object"
  def reveal(): String = privateField + " / " + objectPrivate
}
class SubSecrets extends Secrets {
  def show(): String = protectedField             // OK - subclass can access protected
  // def bad() = privateField                     // ERROR - private not visible here
}

object OOP2 {
  def main(args: Array[String]): Unit = {

    println("---- Abstract class + Inheritance ----")
    val d = new Dog("Rex")
    d.speak(); d.eat()
    val a: Animal = d                             // polymorphism: parent type reference
    a.speak()

    println("---- Field overriding ----")
    println(new Child().info())                   // I am a child

    println("---- Final ----")
    val b = new Base
    println(b.MAX); println(b.locked()); println(b.open())

    println("---- Traits & Mixins ----")
    new Circle(2).run()
    // mix in a trait at object-creation time (anonymous mixin)
    val dogWithLog = new Dog("Fido") with Logger
    dogWithLog.log("Logger mixed into Dog on the fly")

    println("---- Access Modifiers ----")
    val s = new Secrets
    println(s.publicField)
    println(s.reveal())
    // println(s.privateField)    // ERROR (private)
    // println(s.protectedField)  // ERROR (protected, we are outside)
    println(new SubSecrets().show())
  }
}

In [ ]:
!scala-cli run --server=false oop2.scala

## 5. Scala Array & REPL
The **REPL** (Read-Eval-Print Loop) is Scala's interactive shell: type an expression, see the result instantly. Colab cells can't hold an interactive session, but you can still run REPL-style one-liners with `scala-cli run -e` (below). On your own machine use `scala-cli repl`.

In [ ]:
%%writefile arrays.scala
//> using scala 2.13.12
import scala.collection.mutable.ArrayBuffer

object Arrays {
  def main(args: Array[String]): Unit = {
    val a = Array(5, 3, 8, 1)
    println(a.sorted.mkString(","))         // 1,3,5,8   (new sorted array)
    println(a.map(_ * 2).mkString(","))
    println(a.indexOf(8))                   // 2
    a.update(0, 50)                         // same as a(0) = 50
    println(a.mkString(","))

    // 2D array
    val matrix = Array.ofDim[Int](2, 3)     // 2 rows x 3 columns
    matrix(0)(1) = 7
    println(matrix.map(_.mkString(" ")).mkString("\n"))

    // Array.fill / range
    println(Array.fill(3)("x").mkString)
    println(Array.range(0, 10, 2).mkString(","))   // 0,2,4,6,8

    // ArrayBuffer = resizable array
    val buf = ArrayBuffer(1, 2, 3)
    buf += 4; buf.append(5); buf.remove(0)
    println(buf)
  }
}

In [ ]:
!scala-cli run --server=false arrays.scala

In [ ]:
# REPL-style one-liner: evaluates the expression and prints it, like typing it in the REPL
!scala-cli run --server=false -S 2.13.12 -e 'val xs = List(1,2,3); println(xs.map(_ * 10))'

## 6. Scala Libraries: Breeze (numerical computing)

In [ ]:
%%writefile breeze_demo.scala
//> using scala 2.13.12
//> using dep org.scalanlp::breeze:2.1.0

// Breeze = numerical processing library (vectors, matrices, linear algebra, statistics)
import breeze.linalg._
import breeze.numerics._
import breeze.stats._

object BreezeDemo {
  def main(args: Array[String]): Unit = {
    // Vectors
    val v = DenseVector(1.0, 2.0, 3.0)
    val w = DenseVector(4.0, 5.0, 6.0)
    println(v + w)                    // element-wise add
    println(v *:* w)                  // element-wise multiply
    println(v dot w)                  // dot product = 32.0
    println(v * 2.0)                  // scalar multiply
    println(sum(v))                   // 6.0
    println(sqrt(v))                  // numerics: element-wise sqrt

    // Matrices
    val m = DenseMatrix((1.0, 2.0), (3.0, 4.0))
    println(m)
    println(m.t)                      // transpose
    println(m * m)                    // matrix multiplication
    println(det(m))                   // determinant = -2.0
    println(inv(m))                   // inverse
    println(m * DenseVector(1.0, 1.0))

    // Statistics
    val data = DenseVector(2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0)
    println(s"mean=${mean(data)} variance=${variance(data)} stddev=${stddev(data)}")

    // Useful constructors
    println(DenseVector.zeros[Double](3))
    println(DenseMatrix.eye[Double](3))      // identity matrix
    println(linspace(0, 1, 5))               // 0, .25, .5, .75, 1
  }
}

In [ ]:
!scala-cli run --server=false breeze_demo.scala

## 7. Scala Libraries: Saddle (data frames / series)
> Saddle is an older library (like pandas for Scala) and only supports Scala 2.12 and below, so this cell is pinned to Scala 2.12. If Maven resolution ever fails, that's due to the library's age - the rest of the notebook is unaffected.

In [ ]:
%%writefile saddle_demo.scala
//> using scala 2.12.18
//> using dep org.scala-saddle::saddle-core:1.3.4

// Saddle = Vec (1D array), Series (indexed Vec), Frame (table), similar to pandas
import org.saddle._

object SaddleDemo {
  def main(args: Array[String]): Unit = {
    // Vec: typed 1D vector
    val v = Vec(1, 2, 3, 4)
    println(v)
    println(v.sum)
    println(v.mean)

    // Series: Vec with an index (labels)
    val s = Series("a" -> 10, "b" -> 20, "c" -> 30)
    println(s)
    println(s.mapValues(_ * 2))         // transform values
    println(s("b"))                     // lookup by label

    // Frame: table of columns (like a DataFrame)
    val f = Frame("x" -> Series("r1" -> 1, "r2" -> 2),
                  "y" -> Series("r1" -> 3, "r2" -> 4))
    println(f)
    println(f.colAt(0))                 // first column
    println(f.mapValues(_ * 10))
  }
}

In [ ]:
!scala-cli run --server=false saddle_demo.scala

## 8. Exception Handling

In [ ]:
%%writefile exceptions.scala
//> using scala 2.13.12
import scala.util.{Try, Success, Failure}

// Custom exception: extend Exception
class InvalidAgeException(msg: String) extends Exception(msg)

object Exceptions {

  def validate(age: Int): Unit =
    if (age < 0) throw new InvalidAgeException(s"Age $age is invalid")   // 'throw' raises an exception

  def main(args: Array[String]): Unit = {

    // ---------- try / catch / finally ----------
    println("---- try-catch-finally ----")
    try {
      val r = 10 / 0                                  // ArithmeticException
      println(r)
    } catch {
      case e: ArithmeticException => println("Cannot divide by zero: " + e.getMessage)
      case e: Exception           => println("Other error: " + e)   // general fallback
    } finally {
      println("finally ALWAYS runs (cleanup code)")
    }

    // ---------- multiple exception types ----------
    println("---- Multiple cases ----")
    try {
      val arr = Array(1, 2, 3)
      println(arr(5))                                 // ArrayIndexOutOfBoundsException
    } catch {
      case _: ArrayIndexOutOfBoundsException => println("Bad index")
      case _: NullPointerException           => println("Null!")
    }

    try { "abc".toInt } catch {
      case e: NumberFormatException => println("Not a number: " + e.getMessage)
    }

    // ---------- custom exception ----------
    println("---- Custom exception ----")
    try validate(-5) catch { case e: InvalidAgeException => println("Caught: " + e.getMessage) }

    // ---------- Try / Success / Failure (functional style) ----------
    println("---- Try ----")
    val result = Try("42".toInt)
    result match {
      case Success(v) => println(s"Parsed: $v")
      case Failure(e) => println(s"Failed: ${e.getMessage}")
    }
    println(Try("oops".toInt).getOrElse(-1))          // default on failure
    println(Try(10 / 2).map(_ * 2))                   // Success(10)

    // ---------- Option and Either ----------
    println("---- Option / Either ----")
    def safeDiv(a: Int, b: Int): Either[String, Int] =
      if (b == 0) Left("division by zero") else Right(a / b)   // Left = error, Right = success
    println(safeDiv(10, 2))
    println(safeDiv(1, 0))
    println(Some(5).map(_ + 1))                       // Some(6)
    println(None.getOrElse("empty"))
  }
}

In [ ]:
!scala-cli run --server=false exceptions.scala

## 9. Apache Spark: Architecture, Big Data, Components

### Spark architecture (read this first)
```
            +-----------------------+
            |  Driver Program       |   <- your main(); holds SparkSession / SparkContext
            |  (DAG scheduler)      |
            +----------+------------+
                       |  asks for resources
            +----------v------------+
            |   Cluster Manager     |   <- Standalone / YARN / Kubernetes / Mesos / local
            +----+-------------+----+
                 |             |
        +--------v---+   +-----v------+
        | Worker Node|   | Worker Node|
        | Executor   |   | Executor   |   <- run TASKS on partitions of data, cache data
        +------------+   +------------+
```
* **Driver**: runs your code, builds a DAG (plan) of transformations, splits it into stages/tasks.
* **Cluster manager**: allocates CPU/memory. Here we use `local[*]` (all cores of the Colab VM act as the cluster).
* **Executors**: JVM processes on workers that run tasks and store cached data.
* **Lazy evaluation**: *transformations* (map, filter) only build a plan; *actions* (count, collect, show) trigger execution.

### Spark for Big Data
Big data = data too large for one machine. Spark splits it into **partitions**, processes them in parallel in memory, and recovers lost partitions from the lineage (fault tolerance).

### Spark components
| Component | Purpose |
|---|---|
| **Spark Core / RDD** | Basic engine: distributed collections, scheduling |
| **Spark SQL / DataFrame** | Structured data, SQL queries |
| **Structured Streaming** | Real-time stream processing |
| **MLlib** | Machine learning library |
| **GraphX** | Graph processing (graphs, PageRank) |

In [ ]:
%%writefile spark_demo.scala
//> using scala 2.13.12
//> using dep org.apache.spark::spark-sql:3.5.1
//> using dep org.apache.spark::spark-mllib:3.5.1

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.ml.feature.VectorAssembler
import org.apache.spark.ml.regression.LinearRegression

// Case class used to turn data into a typed Dataset (must be defined OUTSIDE main)
case class Employee(name: String, dept: String, salary: Double)

object SparkDemo {
  def main(args: Array[String]): Unit = {

    // ---------- SparkSession: entry point (driver side) ----------
    val spark = SparkSession.builder()
      .appName("ColabSparkDemo")
      .master("local[*]")                       // local mode: use all cores as "executors"
      .config("spark.ui.enabled", "false")
      .getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")     // hide noisy logs
    import spark.implicits._
    val sc = spark.sparkContext                 // SparkContext: entry point for RDDs

    // ================= 1. SPARK CORE (RDD) =================
    println("===== RDD (Spark Core) =====")
    val rdd = sc.parallelize(1 to 10, 4)        // distribute data into 4 partitions
    println("Partitions: " + rdd.getNumPartitions)
    val squares = rdd.map(x => x * x)           // TRANSFORMATION (lazy)
    println(squares.filter(_ % 2 == 0).collect().mkString(","))   // ACTION (runs the job)
    println("Sum = " + rdd.reduce(_ + _))

    // Classic word count (map -> reduceByKey = shuffle across the cluster)
    val lines = sc.parallelize(Seq("big data with spark", "spark is fast", "big data big value"))
    val counts = lines.flatMap(_.split(" ")).map(w => (w, 1)).reduceByKey(_ + _)
    counts.collect().sortBy(-_._2).foreach(println)

    // ================= 2. SPARK SQL / DATAFRAME =================
    println("===== DataFrame / Spark SQL =====")
    val df = Seq(
      Employee("Asha",  "IT",    60000),
      Employee("Ravi",  "IT",    80000),
      Employee("Meena", "HR",    50000),
      Employee("John",  "Sales", 55000),
      Employee("Priya", "Sales", 70000)
    ).toDF()
    df.printSchema()
    df.show()
    df.filter($"salary" > 55000).show()
    df.groupBy("dept").agg(avg("salary").as("avg_salary"), count("*").as("n")).show()

    df.createOrReplaceTempView("emp")           // register as SQL table
    spark.sql("SELECT dept, MAX(salary) AS top FROM emp GROUP BY dept ORDER BY top DESC").show()

    // Typed Dataset
    val ds = df.as[Employee]
    println(ds.map(_.name.toUpperCase).collect().mkString(", "))

    // ================= 3. STRUCTURED STREAMING (concept) =================
    // A streaming DataFrame uses the same API; here a built-in "rate" source generates rows.
    println("===== Structured Streaming =====")
    val stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
    val query = stream.writeStream.format("memory").queryName("rate_tbl").outputMode("append").start()
    Thread.sleep(3000)                          // let it run for 3 seconds
    println("Streamed rows so far: " + spark.sql("SELECT * FROM rate_tbl").count())
    query.stop()

    // ================= 4. MLlib =================
    println("===== MLlib: Linear Regression =====")
    val train = Seq((1.0, 2.0), (2.0, 4.1), (3.0, 5.9), (4.0, 8.2), (5.0, 9.9)).toDF("x", "label")
    val features = new VectorAssembler().setInputCols(Array("x")).setOutputCol("features").transform(train)
    val model = new LinearRegression().setMaxIter(50).fit(features)
    println(s"slope=${model.coefficients(0)}  intercept=${model.intercept}")
    val test = new VectorAssembler().setInputCols(Array("x")).setOutputCol("features")
      .transform(Seq(6.0, 7.0).toDF("x"))
    model.transform(test).select("x", "prediction").show()

    // ================= 5. GraphX =================
    // GraphX (Graph, vertices, edges, PageRank) is Scala/RDD-based. It lives in the
    // spark-graphx module (add: //> using dep org.apache.spark::spark-graphx:3.5.1).

    spark.stop()                                // release resources
  }
}

In [ ]:
!scala-cli run --server=false spark_demo.scala

---
### Quick revision map
| Topic | Section |
|---|---|
| Function format, variables, strings, input, numbers, types, operators, booleans | 1 |
| Array, List, Tuple, Set, HashSet, Map, combinators (map/filter/flatMap/fold) | 2 |
| Class, Object, Anonymous, Singleton, Companion, Case class, Constructors, Overloading, `this` | 3 |
| Inheritance, Field overriding, Final, Abstract class, Trait, Mixins, Access modifiers | 4 |
| Scala Array, REPL | 5 |
| Breeze, Saddle | 6, 7 |
| Exception handling | 8 |
| Spark architecture, Big Data, Components | 9 |